# Variational inference — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Turn inference into optimization over a simpler distribution
Variational inference chooses the closest tractable distribution by maximizing the ELBO. These examples compute an ELBO for categorical q, optimize q, compare KL directions, use mean-field factorization, and show a coordinate update.

In [ ]:
# 1) ELBO = E_q log p - E_q log q for a categorical posterior target
p=normalize([0.1,0.3,0.6]); q=normalize([0.2,0.5,0.3]); elbo=np.sum(q*np.log(p))-np.sum(q*np.log(q))
plt.figure(figsize=(6,3)); plt.bar(['ELBO'],[elbo]); plt.title(f'ELBO={elbo:.3f}')
assert abs(elbo+0.1860980938270007)<1e-12

In [ ]:
# 2) optimal unrestricted q equals normalized p
qs=np.linspace(0.01,0.98,100); vals=[]
for a in qs:
    qq=np.array([a,(1-a)/2,(1-a)/2]); vals.append(np.sum(qq*np.log(p))-np.sum(qq*np.log(qq)))
plt.figure(figsize=(6,3)); plt.plot(qs,vals); plt.axvline(p[0],c='r',ls='--'); plt.title('ELBO peaks at true posterior when family is flexible')
assert abs(qs[int(np.argmax(vals))]-p[0])<0.02

In [ ]:
# 3) KL(q||p) is the gap from log evidence in normalized example
kl=np.sum(q*np.log(q/p)); plt.figure(figsize=(6,3)); plt.bar(['KL(q||p)'],[kl]); plt.title(f'gap={kl:.3f}')
assert abs(kl-0.18609809382700085)<1e-12

In [ ]:
# 4) mean-field approximation cannot represent correlation exactly
joint=normalize([[3,1],[1,3]]); qx=joint.sum(1); qy=joint.sum(0); mf=np.outer(qx,qy)
plt.figure(figsize=(6,3)); plt.imshow(joint-mf); plt.colorbar(); plt.title('correlation left out by factorization')
assert np.max(np.abs(joint-mf))>0.1

In [ ]:
# 5) coordinate update sets a factor to expected log joint
qy=np.array([0.7,0.3]); logj=np.log(joint); logqx=np.array([np.sum(qy*logj[x,:]) for x in [0,1]]); qx_new=softmax(logqx)
plt.figure(figsize=(6,3)); plt.bar(['x=0','x=1'],qx_new); plt.title('one mean-field coordinate update')
assert abs(qx_new.sum()-1)<1e-12